# Mice Protein Expression - Exploratory Data Analysis (EDA)
## Multi-Class Classification for Down's Syndrome Treatment Response

This notebook performs comprehensive EDA on mice protein expression data from IIT Bombay.

## Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## Loading Dataset

In [ ]:
# Download and load dataset
url = 'https://www.ee.iitb.ac.in/~asethi/Dump/MouseTrain.csv'
dataset = pd.read_csv(url)

print(f"Dataset Shape: {dataset.shape}")
print(f"\nFirst 5 rows:")
display(dataset.head())

## Data Overview

In [ ]:
# Basic info
print("Dataset Info:")
print(dataset.info())
print(f"\nDataset Description:")
display(dataset.describe())

## Missing Values Analysis

In [ ]:
# Check for missing values
missing_data = pd.DataFrame({
    'Column': dataset.columns,
    'Missing_Count': dataset.isnull().sum(),
    'Missing_Percentage': (dataset.isnull().sum() / len(dataset) * 100)
})

missing_data = missing_data[missing_data['Missing_Count'] > 0]
if len(missing_data) > 0:
    display(missing_data)
else:
    print("No missing values found!")

## Target Variable Analysis

In [ ]:
# Extract targets
genotype = dataset.iloc[:, -2]  # Binary
treatment = dataset.iloc[:, -1]  # Multi-class

print("Genotype (Binary) Distribution:")
print(genotype.value_counts())
print(f"\nTreatment/Behavior (4-class) Distribution:")
print(treatment.value_counts())

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Binary
genotype.value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Genotype Distribution (Binary)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Genotype')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Multi-class
treatment.value_counts().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Treatment/Behavior Distribution (4-class)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Treatment/Behavior')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Feature Correlation Analysis

In [ ]:
# Get numeric features only
numeric_data = dataset.select_dtypes(include=[np.number])

# Calculate correlation matrix
corr_matrix = numeric_data.corr(method='spearman')

# Plot heatmap
plt.figure(figsize=(20, 18))
sns.heatmap(np.abs(corr_matrix), cmap='coolwarm', square=True, cbar_kws={"shrink": 0.8})
plt.title('Spearman Correlation Matrix (Absolute Values)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nHigh correlations (>= 0.7):")

In [ ]:
# Find highly correlated pairs
def find_highly_correlated(corr_matrix, threshold=0.7):
    pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) >= threshold:
                pairs.append({
                    'Feature_1': corr_matrix.columns[i],
                    'Feature_2': corr_matrix.columns[j],
                    'Correlation': corr_matrix.iloc[i, j]
                })
    return pd.DataFrame(pairs)

high_corr = find_highly_correlated(corr_matrix, threshold=0.7)
if len(high_corr) > 0:
    print(f"Found {len(high_corr)} highly correlated pairs")
    display(high_corr.sort_values('Correlation', ascending=False))
else:
    print("No highly correlated pairs found")

## Features Distribution

In [ ]:
# Plot distributions of first 12 features
features = dataset.iloc[:, :-2].columns[:12]
fig, axes = plt.subplots(4, 3, figsize=(16, 12))
axes = axes.ravel()

for i, feature in enumerate(features):
    axes[i].hist(dataset[feature], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{feature}', fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Feature Statistics

In [ ]:
# Get feature statistics
features_only = dataset.iloc[:, :-2]

stats_df = pd.DataFrame({
    'Mean': features_only.mean(),
    'Std': features_only.std(),
    'Min': features_only.min(),
    'Max': features_only.max(),
    'Missing': features_only.isnull().sum()
})

print("Feature Statistics:")
display(stats_df.head(15))

## Data Preprocessing Overview

In [ ]:
# Summary of preprocessing steps
print("""\n=== PREPROCESSING STEPS ===")
print(f"1. Original Features: {dataset.shape[1] - 2}")
print(f"2. Missing Values: Will be imputed using mean strategy")
print(f"3. Highly Correlated Features: Will be removed (ITSN1_N, NR2A_N, pBRAF_N, ERK_N)")
print(f"4. PCA Reduction: 77 features → 37 components")
print(f"5. Label Encoding: Categorical targets → Numeric")
print(f"\n=== TARGET VARIABLES ===")
print(f"Target 1 (Binary): Genotype - {genotype.unique()}")
print(f"Target 2 (Multi-class): Treatment/Behavior - {treatment.unique()}")

## Summary Statistics

In [ ]:
print(f"""\n=== DATASET SUMMARY ===")
print(f"Total Samples: {len(dataset)}")
print(f"Total Features (Proteins): {dataset.shape[1] - 2}")
print(f"Features with missing values: {dataset.isnull().sum().sum()}")
print(f"\n=== CLASS DISTRIBUTION ===")
print(f"Genotype (Binary) Classes: {len(genotype.unique())}")
print(f"  - Class Distribution:")
for val, count in genotype.value_counts().items():
    print(f"    {val}: {count} samples ({count/len(genotype)*100:.1f}%)")

print(f"\nTreatment/Behavior (Multi-class) Classes: {len(treatment.unique())}")
print(f"  - Class Distribution:")
for val, count in treatment.value_counts().items():
    print(f"    {val}: {count} samples ({count/len(treatment)*100:.1f}%)")

## Next Steps

1. **Train Models**: Run `python -m app.train` to train all models
2. **Start API**: Run `uvicorn app.main:app --reload` to start FastAPI
3. **Make Predictions**: Use the API endpoints to make predictions on new data